In [25]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, Tuple
import tempfile
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import ChromBERT utilities
import css_utility as crb

In [26]:
# essential data
LTR="../database/LTR/hg38_RepeatMasker_LTR.bed"

# Liver Cell Line (E066 from ROADMAP)
liver_cell = "../database/ROADMAP/hg38_mnemonics/unzipped/E066_15_coreMarks_hg38lift_mnemonics.bed"

# HepG2 Hepatocellular Carcinoma Cell Line (E118 from ROADMAP)
hepg2_cell = "../database/ROADMAP/hg38_mnemonics/unzipped/E118_15_coreMarks_hg38lift_mnemonics.bed"

In [27]:
def make_ltr(inp="../database/LTR/hg38_RepeatMasker_LTR.bed",
                   out="./hg38_RepeatMasker_LTR_pm1kb.bed", dist=1000):
    
    # Read as 6-column BED
    df_ltr = pd.read_csv(
        inp,
        sep="\t",
        header=None,
        names=["chrom", "start", "end", "name", "score", "strand"],
        comment="#",
        engine="python"
    )

    # Keep only real BED rows
    df_ltr = df_ltr[df_ltr["chrom"].astype(str).str.startswith("chr")].copy()

    # Ensure numeric types
    df_ltr["start"] = pd.to_numeric(df_ltr["start"], errors="coerce")
    df_ltr["end"]   = pd.to_numeric(df_ltr["end"], errors="coerce")
    df_ltr = df_ltr.dropna(subset=["start", "end"])

    df_ltr["start"] = df_ltr["start"].astype(int)
    df_ltr["end"]   = df_ltr["end"].astype(int)

    # Expand ±1 kb
    df_ltr["start"] = (df_ltr["start"] - dist).clip(lower=0)
    df_ltr["end"]   = df_ltr["end"] + dist

    # Save BED
    df_ltr.to_csv(out, sep="\t", header=False, index=False)

    return df_ltr

In [34]:
df_ltr = make_ltr(
    inp=LTR,
    out="hg38_RepeatMasker_LTR_pm1kb.bed",
    dist=1000
)

In [35]:
df_ltr.head()

,chrom,start,end,name,score,strand
1,chr1,159382130,159385355,MSTA-int,6837.0,+
2,chr1,226491179,226493451,HUERS-P3b-int,2094.0,-
3,chr1,6290167,6292531,MLT1C,875.0,+
4,chr1,26213342,26215429,MLT1K,365.0,-
5,chr1,29358332,29361290,LTR5_Hs,8320.0,+


In [36]:
def make_ltr_df_on_celltype_from_mnemonics(
    mnemonics_bed,
    ltr_df,
    canonical_chroms=None,
    chrom_order=None,
    bin_size=200
):
    if canonical_chroms is None:
        canonical_chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

    if chrom_order is None:
        chrom_order = {f"chr{i}": i for i in range(1, 23)}
        chrom_order.update({"chrX": 23, "chrY": 24})

    # 1. read original mnemonics BED
    df_raw = pd.read_csv(
        mnemonics_bed,
        sep="\t",
        header=None,
        names=["chromosome", "start", "end", "state"],
        usecols=[0, 1, 2, 3]
    )

    # 2. keep canonical chromosomes
    df_raw = df_raw[df_raw["chromosome"].isin(canonical_chroms)].copy()

    # 3. numeric cleanup
    df_raw["start"] = pd.to_numeric(df_raw["start"], errors="coerce")
    df_raw["end"] = pd.to_numeric(df_raw["end"], errors="coerce")
    df_raw = df_raw.dropna(subset=["start", "end"]).copy()
    df_raw["start"] = df_raw["start"].astype(int)
    df_raw["end"] = df_raw["end"].astype(int)

    # 4. sort in Python
    df_raw["chrom_order"] = df_raw["chromosome"].map(chrom_order)
    df_raw = (
        df_raw
        .sort_values(["chrom_order", "start", "end"])
        .drop(columns="chrom_order")
        .reset_index(drop=True)
    )

    # 5. convert mnemonic -> state number only
    #    e.g. 15_Quies -> 15
    df_raw["state"] = df_raw["state"].astype(str).str.replace(r"_.*", "", regex=True)

    # 6. write temporary stateno BED for crb.bed2df_expanded()
    with tempfile.NamedTemporaryFile(mode="w", suffix=".stateno.bed", delete=False) as tmp:
        tmp_path = tmp.name
        df_raw.to_csv(tmp_path, sep="\t", header=False, index=False)

    try:
        # 7. use YOUR function here
        df_state = crb.bed2df_expanded(tmp_path)

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

    # 8. keep canonical chromosomes and sort
    df_state = (
        df_state[df_state["chromosome"].isin(canonical_chroms)]
        .assign(chrom_order=lambda x: x["chromosome"].map(chrom_order))
        .sort_values(["chrom_order", "start", "end"])
        .drop(columns="chrom_order")
        .reset_index(drop=True)
    )

    # 9. build chromosome-wide state strings from state_seq_full
    chr_seq_map = (
        df_state.groupby("chromosome", sort=False)["state_seq_full"]
        .apply("".join)
        .to_dict()
    )

    # 10. slice onto LTR windows
    def slice_chr_seq(chrom, start_bp, end_bp):
        seq = chr_seq_map.get(chrom)
        if seq is None:
            return None

        bin_start = start_bp // bin_size
        bin_end = (end_bp - 1) // bin_size + 1
        return seq[bin_start:bin_end]

    df_ltr_on_cell = ltr_df.copy()
    df_ltr_on_cell = df_ltr_on_cell[df_ltr_on_cell["chrom"].isin(canonical_chroms)].copy()

    df_ltr_on_cell["length"] = df_ltr_on_cell["end"] - df_ltr_on_cell["start"]

    df_ltr_on_cell["state_seq_full_slice"] = df_ltr_on_cell.apply(
        lambda r: slice_chr_seq(r["chrom"], int(r["start"]), int(r["end"])),
        axis=1
    )

    df_ltr_on_cell["n_bins"] = df_ltr_on_cell["state_seq_full_slice"].str.len()

    df_ltr_on_cell["chrom_order"] = df_ltr_on_cell["chrom"].map(chrom_order)

    df_ltr_on_cell = (
        df_ltr_on_cell
        .sort_values(["chrom_order", "start", "end"])
        .drop(columns="chrom_order")
        .reset_index(drop=True)
    )

    df_ltr_on_cell = df_ltr_on_cell[df_ltr_on_cell["n_bins"] > 0]

    return df_ltr_on_cell

In [37]:
df_ltr_E066 = make_ltr_df_on_celltype_from_mnemonics(
    liver_cell,
    df_ltr
)

In [38]:
df_ltr_E066

,chrom,start,end,name,score,strand,length,state_seq_full_slice,n_bins
0,chr1,20948,23075,MLT1K,254.0,+,2127,OOOOOOOOOOOO,12
1,chr1,29693,31848,MLT1A,741.0,+,2155,OOOOOOOOOOOO,12
2,chr1,29952,32131,MLT1A,741.0,+,2179,OOOOOOOOOOOO,12
3,chr1,33564,35921,MLT1J2,850.0,-,2357,OOOOOOOOOOOOO,13
4,chr1,37255,40464,MLT1E1A-int,3877.0,+,3209,OOOOOOOOOOOOOOOOO,17
...,...,...,...,...,...,...,...,...,...
719759,chrY,25239006,25241335,MER66-int,1531.0,+,2329,IIIIIIIIIIII,12
719760,chrY,25239335,25241651,MER66B,1077.0,+,2316,IIIIIIIIIIIII,13
719761,chrY,25239698,25242327,LTR12D,3938.0,-,2629,IIIIIIIIIIIIII,14
719762,chrY,25240331,25242491,LTR12D,909.0,-,2160,IIIIIIIIIIII,12


In [39]:
df_ltr_E118 = make_ltr_df_on_celltype_from_mnemonics(
    hepg2_cell,
    df_ltr
)
df_ltr_E118

,chrom,start,end,name,score,strand,length,state_seq_full_slice,n_bins
0,chr1,20948,23075,MLT1K,254.0,+,2127,OOOOOOOOOOOO,12
1,chr1,29693,31848,MLT1A,741.0,+,2155,OOOOOOOOOOOO,12
2,chr1,29952,32131,MLT1A,741.0,+,2179,OOOOOOOOOOOO,12
3,chr1,33564,35921,MLT1J2,850.0,-,2357,OOOOOOOOOOOOO,13
4,chr1,37255,40464,MLT1E1A-int,3877.0,+,3209,OOOOOOOOOOOOOOOOO,17
...,...,...,...,...,...,...,...,...,...
719759,chrY,25239006,25241335,MER66-int,1531.0,+,2329,IIIIIIIIIIII,12
719760,chrY,25239335,25241651,MER66B,1077.0,+,2316,IIIIIIIIIIIII,13
719761,chrY,25239698,25242327,LTR12D,3938.0,-,2629,IIIIIIIIIIIIII,14
719762,chrY,25240331,25242491,LTR12D,909.0,-,2160,IIIIIIIIIIII,12


In [40]:
def make_ltr_transition_df(
    df_ltr_on_cell,
    remove_all_O=True,
    min_length_bp=1
):
    """
    Build an analysis-ready LTR transition dataframe from a sliced LTR dataframe.

    Parameters
    ----------
    df_ltr_on_cell : pd.DataFrame
        Output of make_ltr_df_on_celltype_from_mnemonics(), expected to contain:
        chrom, start, end, name, score, strand, length, state_seq_full_slice, n_bins

    remove_all_O : bool, default True
        If True, remove rows whose state_seq_full_slice contains only 'O'.

    min_length_bp : int, default 1
        Minimum length required to compute transition density.

    Returns
    -------
    df_ltr_trans : pd.DataFrame
        Dataframe with additional columns:
        state_comp, n_states, n_transitions, transition_density_per_kb
    """

    df_ltr_trans = df_ltr_on_cell.copy()

    # state composition
    df_ltr_trans["state_comp"] = df_ltr_trans["state_seq_full_slice"].apply(
        lambda s: Counter(s) if isinstance(s, str) else {}
    )

    # optionally remove all-O rows
    if remove_all_O:
        df_ltr_trans = df_ltr_trans[
            df_ltr_trans["state_comp"].apply(lambda c: any(k != "O" for k in c))
        ].copy()

    # distinct number of states
    df_ltr_trans["n_states"] = df_ltr_trans["state_comp"].apply(len)

    # transition count
    def count_transitions(seq):
        if not isinstance(seq, str) or len(seq) == 0:
            return 0
        return sum(seq[i] != seq[i - 1] for i in range(1, len(seq)))

    df_ltr_trans["n_transitions"] = df_ltr_trans["state_seq_full_slice"].apply(count_transitions)

    # transition density per kb
    df_ltr_trans = df_ltr_trans[df_ltr_trans["length"] >= min_length_bp].copy()
    df_ltr_trans["transition_density_per_kb"] = (
        df_ltr_trans["n_transitions"] / (df_ltr_trans["length"] / 1000)
    )

    return df_ltr_trans

In [53]:
df_ltr_trans_E066=make_ltr_transition_df(df_ltr_E066)
df_sorted_E066 = df_ltr_trans_E066.sort_values("transition_density_per_kb", ascending=False)
df_sorted_E066.head(10)

,chrom,start,end,name,score,strand,length,state_seq_full_slice,n_bins,state_comp,n_states,n_transitions,transition_density_per_kb
590929,chr17,30658947,30660993,MLT1G1,409.0,-,2046,GBGGBABAAGO,11,"{'G': 4, 'B': 3, 'A': 3, 'O': 1}",4,8,3.910068
267644,chr6,28753325,28755377,THE1D,1946.0,-,2052,NGBABABBBGE,11,"{'N': 1, 'G': 2, 'B': 5, 'A': 2, 'E': 1}",5,8,3.898635
599789,chr17,76862366,76864714,MLT1C,1106.0,+,2348,GGBBBABABABGE,13,"{'G': 3, 'B': 6, 'A': 3, 'E': 1}",4,9,3.833049
12372,chr1,55820554,55822919,MER57B1,2346.0,+,2365,BGGBGBGGBGBGG,13,"{'B': 5, 'G': 8}",2,9,3.805497
77711,chr2,106416312,106418341,MLT2A1,2529.0,+,2029,GBBAABABGOO,11,"{'G': 2, 'B': 4, 'A': 3, 'O': 2}",4,7,3.449975
565496,chr16,2870987,2873323,MLT2F,440.0,-,2336,EEBAABBABAABG,13,"{'E': 2, 'B': 5, 'A': 5, 'G': 1}",4,8,3.424658
598733,chr17,72047939,72050298,MLT1A0,2011.0,+,2359,GGBBGEEABABAA,13,"{'G': 3, 'B': 4, 'E': 2, 'A': 4}",4,8,3.391267
26012,chr1,108607742,108610101,MLT1A0,1607.0,-,2359,BABABBGBGGEEE,13,"{'B': 5, 'A': 2, 'G': 3, 'E': 3}",4,8,3.391267
107198,chr2,226142758,226144825,MLT2A1,412.0,-,2067,BBGGBGBGGBGG,12,"{'B': 5, 'G': 7}",2,7,3.386551
77710,chr2,106415909,106418273,MLT2A1,2529.0,+,2364,OOGBBAABABGOO,13,"{'O': 4, 'G': 2, 'B': 4, 'A': 3}",4,8,3.384095


In [52]:
df_ltr_trans_E118=make_ltr_transition_df(df_ltr_E118)
df_sorted_E118 = df_ltr_trans_E118.sort_values("transition_density_per_kb", ascending=False)
df_sorted_E118.head(10)

,chrom,start,end,name,score,strand,length,state_seq_full_slice,n_bins,state_comp,n_states,n_transitions,transition_density_per_kb
477451,chr12,54446160,54448203,LTR67B,684.0,+,2043,LJKLKLKLKKLL,12,"{'L': 6, 'J': 1, 'K': 5}",3,9,4.405286
477450,chr12,54445905,54448018,LTR67B,455.0,+,2113,LLJKLKLKLKKL,12,"{'L': 6, 'J': 1, 'K': 5}",3,9,4.259347
378213,chr9,6705758,6707836,MER70B,2171.0,+,2078,BKJJJKKBGBGO,12,"{'B': 3, 'K': 3, 'J': 3, 'G': 2, 'O': 1}",5,8,3.849856
293760,chr6,140512760,140515106,MLT1A0,1640.0,+,2346,GBGBBKJJJAKBG,13,"{'G': 3, 'B': 4, 'K': 2, 'J': 3, 'A': 1}",5,9,3.836317
568764,chr16,14814855,14817028,LTR40a,1681.0,-,2173,EGBGEAEEEAEE,12,"{'E': 7, 'G': 2, 'B': 1, 'A': 2}",4,8,3.681546
332946,chr7,135863400,135865617,MSTD,1739.0,-,2217,BABABBABBAAB,12,"{'B': 7, 'A': 5}",2,8,3.608480
78341,chr2,108412796,108415025,MLT1J1,408.0,+,2229,LMMMMJLKLKLLN,13,"{'L': 5, 'M': 4, 'J': 1, 'K': 2, 'N': 1}",5,8,3.589053
589213,chr17,17468929,17471175,MSTB,1940.0,-,2246,GBGGBLKLGGGL,12,"{'G': 6, 'B': 2, 'L': 3, 'K': 1}",4,8,3.561888
619881,chr19,1385246,1387499,MLT1B,1510.0,-,2253,ABBAAAJKJKJM,12,"{'A': 4, 'B': 2, 'J': 3, 'K': 2, 'M': 1}",5,8,3.550821
259118,chr5,177028755,177031066,MER21B,1924.0,+,2311,BGBGGGLLKKBGE,13,"{'B': 3, 'G': 5, 'L': 2, 'K': 2, 'E': 1}",5,8,3.461705
